In [0]:
%pip install fastmcp pydantic

dbutils.library.restartPython()

In [0]:
import asyncio
import json
import sys
import os

# 1. Map your directory paths perfectly to the absolute project root folder
git_folder_path = os.path.abspath('./')
if git_folder_path not in sys.path:
    sys.path.insert(0, git_folder_path)

# Add the source directory to ensure all nested internal modules resolve cleanly
src_path = os.path.join(git_folder_path, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 2. Set up the sandbox server environment using the new custom package path logic
from fastmcp import FastMCP
from payroll_mcp.tools import payroll_compliance_tool

# Spin up our test sandbox server instance and register your tool
mcp_sandbox = FastMCP("TestServer")
payroll_compliance_tool.register(mcp_sandbox)

# 3. Simulate an AI client testing the tool using the official 'call_tool' method across multiple scenarios
async def execute_test():
    scenarios = [
        ## SCENARIO 1 (PASS - PHILADELPHIA): Mid-grade employee (GS-13, Step 5) in Philadelphia.
        ## Tests multi-city lookup and verifies base + overtime stays safely under Philadelphia's biweekly cap.
        {
            "name": "PASS 1: Philadelphia GS-13 Step 5 (Within Cap)",
            "args": {
                "grade": 13,
                "step": 5,
                "locality": "Philadelphia",
                "premium_pay_earned": 1200.00
            }
        },
        
        ## SCENARIO 2 (PASS - REST OF US): Entry/mid-grade employee (GS-9, Step 3) in Rest-of-US.
        ## Tests lower-locality pay tables and verifies compliance when total gross is well below EX-V.
        {
            "name": "PASS 2: Rest-of-US GS-9 Step 3 (Within Cap)",
            "args": {
                "grade": 9,
                "step": 3,
                "locality": "Rest-of-US",
                "premium_pay_earned": 650.00
            }
        },
        
        ## SCENARIO 3 (FAIL - CAP EXCEEDED IN SAN FRANCISCO): Senior employee (GS-15, Step 8) in San Francisco.
        ## Tests high-locality cap math and confirms EXCEEDS_CAP triggers when overtime pushes gross over the limit.
        {
            "name": "FAIL 1: San Francisco GS-15 Step 8 (Biweekly Cap Exceeded)",
            "args": {
                "grade": 15,
                "step": 8,
                "locality": "San Francisco",
                "premium_pay_earned": 2200.00
            }
        },
        
        ## SCENARIO 4 (FAIL - INVALID GRADE): Out-of-bounds GS grade (Grade 18).
        ## Tests Pydantic boundary validation to confirm invalid grades are rejected cleanly.
        {
            "name": "FAIL 2: Invalid GS Grade 18 (Validation Error Check)",
            "args": {
                "grade": 18,
                "step": 1,
                "locality": "Washington-DC",
                "premium_pay_earned": 500.00
            }
        },
        
        ## SCENARIO 5 (FAIL - UNSUPPORTED CITY): Unrecognized locality name ('Atlantis').
        ## Tests helpful error feedback so the AI agent knows which valid OPM localities are supported.
        {
            "name": "FAIL 3: Unsupported Locality ('Atlantis')",
            "args": {
                "grade": 12,
                "step": 4,
                "locality": "Atlantis",
                "premium_pay_earned": 500.00
            }
        }
    ]

    print("=== 2026 MULTI-CITY PAYROLL COMPLIANCE TEST SUITE ===\n")

    for i, scenario in enumerate(scenarios, 1):
        print(f"[{i}/{len(scenarios)}] {scenario['name']}")
        print(f"Inputs: {scenario['args']}")
        
        try:
            # FastMCP uses call_tool to execute a registered capability by name
            result = await mcp_sandbox.call_tool(
                "validate_payroll_compliance",
                arguments=scenario["args"]
            )
            # Print the clean, formatted response back on your screen!
            print("Output:")
            print(json.dumps(result.structured_content, indent=2))
        except Exception as e:
            print(f"Caught Expected Error:\n{type(e).__name__}: {e}")
            
        print("-" * 50 + "\n")

# Trigger the evaluation loop
await execute_test()


In [0]:
  %sh grep -rn "example_server"
/Workspace/Users/ciara.reed@gsa.gov/mcp-hackathon-template
--include="*.py" --include="*.toml" --include="*.yaml"
--include="*.yml" --include="Dockerfile" --include="*.ipynb"

In [0]:
%sh pip install -e .
dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()